# Alpha EAI - Panel of Experts 训练

**百度飞桨 AI Studio**: aistudio.baidu.com → 新建项目 → 上传此文件 → 环境选 Paddle 2.6 (带 GPU)

架构：4 个专家 × 5 层 Transformer + 6 层后处理 = ~92M 参数

In [ ]:
!pip install datasets transformers accelerate torch -q

In [ ]:
import os
import sys

# HF 国内镜像
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, GPT2Model, GPT2Config, get_linear_schedule_with_warmup
from datasets import load_dataset
from tqdm import tqdm

In [ ]:
class Router(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate_linear = nn.Linear(d_model, num_experts)

    def forward(self, hidden_states):
        logits = self.gate_linear(hidden_states)
        weights = F.softmax(logits, dim=-1)
        top_weights, top_indices = torch.topk(weights, self.top_k, dim=-1)
        top_weights = F.normalize(top_weights, p=1, dim=-1)
        return top_weights, top_indices


class Expert(nn.Module):
    def __init__(self, num_layers=None, d_model=None, n_head=None, d_ff=None):
        super().__init__()
        hf_config = GPT2Config(
            vocab_size=50257,
            n_layer=num_layers or 5,
            n_embd=d_model or 256,
            n_head=n_head or 4,
            n_inner=d_ff or 512,
            use_cache=False,
        )
        self.transformer = GPT2Model(hf_config)
        self.d_model = self.transformer.config.n_embd

    def forward(self, input_embeds, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones(input_embeds.shape[:2], device=input_embeds.device, dtype=torch.long)
        outputs = self.transformer(inputs_embeds=input_embeds, attention_mask=attention_mask, return_dict=True)
        return outputs.last_hidden_state


class ExpertFusion(nn.Module):
    def __init__(self, num_experts, d_model, n_head=4):
        super().__init__()
        self.expert_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_head, batch_first=True)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, expert_outputs, attention_mask=None):
        B, S, N, D = expert_outputs.shape
        expert_outputs_2d = expert_outputs.reshape(B * S, N, D)
        attn_output, _ = self.expert_attention(expert_outputs_2d, expert_outputs_2d, expert_outputs_2d, need_weights=False)
        fused = attn_output.mean(dim=1)
        fused = self.ln(fused)
        return fused.reshape(B, S, D)


class PostProcessing(nn.Module):
    def __init__(self, num_layers=6, d_model=None, n_head=None, d_ff=None):
        super().__init__()
        hf_config = GPT2Config(
            vocab_size=50257,
            n_layer=num_layers,
            n_embd=d_model,
            n_head=n_head or 4,
            n_inner=d_ff or 512,
            use_cache=False,
        )
        self.transformer = GPT2Model(hf_config)
        self.d_model = self.transformer.config.n_embd

    def forward(self, input_embeds, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones(input_embeds.shape[:2], device=input_embeds.device, dtype=torch.long)
        outputs = self.transformer(inputs_embeds=input_embeds, attention_mask=attention_mask, return_dict=True)
        return outputs.last_hidden_state


class PoEModel(nn.Module):
    def __init__(self, num_experts=4, expert_num_layers=5, post_processing_num_layers=6,
                 d_model=256, n_head=4, d_ff=512, top_k=2,
                 vocab_size=50257, max_seq_len=256):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.max_seq_len = max_seq_len

        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(0.1)

        self.router = Router(d_model, num_experts, top_k)

        self.experts = nn.ModuleList([
            Expert(num_layers=expert_num_layers, d_model=d_model, n_head=n_head, d_ff=d_ff)
            for _ in range(num_experts)
        ])

        self.fusion = ExpertFusion(num_experts, d_model, n_head)
        self.post_processing = PostProcessing(num_layers=post_processing_num_layers, d_model=d_model, n_head=n_head, d_ff=d_ff)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def forward(self, input_ids, attention_mask=None, labels=None):
        B, S = input_ids.shape
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        pos_ids = torch.arange(S, device=input_ids.device).unsqueeze(0).expand(B, -1)
        x = self.wte(input_ids) + self.wpe(pos_ids)
        x = self.dropout(x)

        expert_outputs = [expert(x, attention_mask) for expert in self.experts]
        stacked = torch.stack(expert_outputs, dim=2)
        fused = self.fusion(stacked, attention_mask)
        pp_out = self.post_processing(fused, attention_mask)
        logits = self.lm_head(pp_out)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {"loss": loss, "logits": logits}

In [ ]:
class TinyStoriesDataset(Dataset):
    def __init__(self, samples, tokenizer, max_seq_len=256):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text = self.samples[idx]
        encoded = self.tokenizer(text, return_tensors="pt", max_length=self.max_seq_len, truncation=True)
        return encoded["input_ids"].squeeze(0)


def collate_fn(batch, pad_value=0):
    max_len = max(x.size(0) for x in batch)
    padded, masks = [], []
    for x in batch:
        pad_len = max_len - x.size(0)
        padded.append(F.pad(x, (0, pad_len), value=pad_value))
        masks.append(torch.cat([torch.ones(x.size(0), dtype=torch.long), torch.zeros(pad_len, dtype=torch.long)]))
    input_ids = torch.stack(padded)
    attention_mask = torch.stack(masks)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": input_ids.clone()}


def load_data(batch_size=16, max_seq_len=256, num_samples=10000):
    print("Loading TinyStories (via mirror)...")
    raw = load_dataset("the_cool_kid/tiny_stories", split="train", streaming=True)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    
    print(f"Collecting {num_samples} samples...")
    samples = []
    for i, item in enumerate(raw):
        samples.append(item["text"])
        if len(samples) >= num_samples:
            break
    
    ds = TinyStoriesDataset(samples, tokenizer, max_seq_len)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0)
    print(f"Dataset: {len(ds)} samples, {len(loader)} batches")
    return ds, loader, tokenizer

In [ ]:
def train(model, dataloader, num_epochs=3, lr=3e-4, weight_decay=0.01, warmup_ratio=0.05, save_dir="output"):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    model.to(device)
    
    no_decay = ["bias", "LayerNorm.weight"]
    param_groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(param_groups, lr=lr)
    total_steps = len(dataloader) * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch} avg loss: {avg_loss:.4f}")
        path = os.path.join(save_dir, f"poe_epoch_{epoch}.pt")
        torch.save(model.state_dict(), path)
        print(f"Saved: {path}")
    return model


def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids)
            logits = outputs["logits"][:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(input_ids.squeeze(), skip_special_tokens=True)

In [ ]:
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

ds, train_loader, tokenizer = load_data(batch_size=16, max_seq_len=256, num_samples=10000)

model = PoEModel(
    num_experts=4, expert_num_layers=5, post_processing_num_layers=6,
    d_model=256, n_head=4, d_ff=512, top_k=2, max_seq_len=256,
)
total = sum(p.numel() for p in model.parameters())
expert_p = sum(p.numel() for e in model.experts for p in e.parameters())
pp_p = sum(p.numel() for p in model.post_processing.parameters())
print(f"Total params: {total:,}")
print(f"  Experts (4x): {expert_p:,}")
print(f"  Post-processing: {pp_p:,}")

model = train(model, train_loader, num_epochs=3, lr=3e-4)

In [ ]:
print(generate(model, tokenizer, "Once upon a time", max_new_tokens=100, temperature=0.8))
print(generate(model, tokenizer, "There was a little", max_new_tokens=100, temperature=0.8))